# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Colab/Kaggle: repo đã `git clone` vào thư mục hiện tại. Máy cá nhân: kernel chạy từ submissions/<...>/code/.
REPO_URL = "https://github.com/tam253211-a11y/K4-Track4-Day2-DangHuuTam-2A202602940-Deeplearning-Advance.git"
REPO_DIR = "K4-Track4-Day2-DangHuuTam-2A202602940-Deeplearning-Advance"
if not os.path.exists(f"{REPO_DIR}/eval.py") and os.path.exists("../../../eval.py"):
    REPO_DIR = "../../.."
if not os.path.exists(f"{REPO_DIR}/eval.py"):    # Colab/Kaggle mở notebook lẻ: clone repo về
    !git clone -q {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{REPO_DIR}/eval.py"), "Không tìm thấy gốc repo, sửa REPO_DIR"
os.chdir(REPO_DIR)                           # từ đây mọi đường dẫn (data/, predictions/...) tính từ gốc repo
REPO_DIR = os.getcwd()
CODE_DIR = f"{REPO_DIR}/submissions/2A202602940_dang_huu_tam/code"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, CODE_DIR)                 # code/ đã chép từ starter/

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/images/   # zip không có thư mục con: .jpg nằm thẳng trong data/images/
!ls data/images | head -3; ls data/images | wc -l

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

### Nơi lưu bền: Google Drive
`/content` bị xoá khi Colab ngắt phiên. Ô dưới nối `runs/` (checkpoint, log, logit), `predictions/`, `curves/`, `eval_out/`
sang Drive, nên mọi thứ code ghi vào các thư mục này nằm thẳng trên Drive. Chạy lại ô này sau mỗi lần phiên khởi động lại.

In [ ]:
OUTPUT_DIRS = ["runs", "predictions", "curves", "eval_out"]
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/K4-Day2-DeepWeeds"
elif os.path.isdir("/kaggle/working"):
    PERSIST_DIR = "/kaggle/working"           # Kaggle: lưu thành output khi Save & Run All
else:
    PERSIST_DIR = None                        # máy cá nhân: ổ đĩa đã bền sẵn
if PERSIST_DIR:
    for d in OUTPUT_DIRS:
        target = f"{PERSIST_DIR}/{d}"
        os.makedirs(target, exist_ok=True)
        if not os.path.islink(d):
            assert not os.path.exists(d) or not os.listdir(d), f"{d}/ đã có dữ liệu cục bộ, chép sang {target} trước"
            if os.path.exists(d):
                os.rmdir(d)
            os.symlink(target, d)
for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"{d:12s} -> {os.path.realpath(d)}")


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import dataset
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

# TODO: vẽ biểu đồ phân bố lớp; xem >= 3 ảnh mỗi lớp; đối chiếu với Table 1 của bài báo
#       (Negatives 9.106; các loài 1.009-1.125 ảnh).


In [ ]:
# Kiểm tra model.py: tag trọng số, #params, GMAC, nhóm tham số, đóng băng, loss ban đầu ~ ln 9
import importlib, math, torch, model as M
importlib.reload(M)
rows = []
for key in M.SUGGESTED_BACKBONES:
    m = M.build_model(key, init="finetune")
    groups = {g["name"]: sum(p.numel() for p in g["params"]) for g in M.param_groups(m, 1e-4, 1e-3, 0.05)}
    rows.append({"backbone": key, "tag": M.weight_tag(m), "params_M": round(M.count_params(m), 2),
                 "GMAC": round(M.count_gmacs(m, 224), 2), **{f"n_{k}": v for k, v in groups.items()}})
    del m
display(pd.DataFrame(rows))

# đóng băng: chỉ head còn requires_grad, BN của backbone ở eval sau set_train_mode
m = M.build_model("resnet50", init="frozen")
M.set_train_mode(m)
bn_train = [n for n, x in m.named_modules() if isinstance(x, torch.nn.BatchNorm2d) and x.training]
trainable = [n for n, p in m.named_parameters() if p.requires_grad]
print("frozen: trainable =", trainable, "| BN còn train mode:", len(bn_train))
assert trainable == ["fc.weight", "fc.bias"] and not bn_train

# loss CE ban đầu với head mới phải gần -ln(1/9) = 2.197
m = M.build_model("resnet50").eval()
x = torch.randn(32, 3, 224, 224); y = torch.randint(0, 9, (32,))
with torch.no_grad():
    loss0 = torch.nn.functional.cross_entropy(m(x), y).item()
print(f"loss ban đầu = {loss0:.3f} (kỳ vọng ~ {math.log(9):.3f})")


In [ ]:
# Kiểm tra losses.py (GUIDE mục 3.2): focal gamma=0 == CE, label smoothing eps=0 == CE, CutMix lam theo diện tích thật
import losses as L
importlib.reload(L)
torch.manual_seed(0)
logits, y = torch.randn(64, 9), torch.randint(0, 9, (64,))
ce = torch.nn.functional.cross_entropy(logits, y)
assert abs(L.FocalLoss(gamma=0)(logits, y) - ce) < 1e-6, "focal gamma=0 phải bằng CE"
assert abs(L.LabelSmoothingCE(0.0)(logits, y) - ce) < 1e-6, "LS eps=0 phải bằng CE"
assert abs(L.LabelSmoothingCE(0.1)(logits, y) - torch.nn.functional.cross_entropy(logits, y, label_smoothing=0.1)) < 1e-6
print("focal(gamma=2) =", L.FocalLoss(2.0)(logits, y).item(), "< CE =", ce.item())
w = L.class_weights(np.bincount(train_df.Label, minlength=9)); print("class weights (1/n):", w.numpy().round(3))
x = torch.zeros(8, 3, 224, 224); x[4:] = 1
xm, (ya, yb, lam) = L.mix_batch(x, torch.arange(8), 1.0, "cutmix")
print(f"CutMix lam = {lam:.3f}; tỉ lệ pixel giữ nguyên ảnh 0 = {(xm[0] == x[0]).float().mean():.3f} (>= lam)")
print("OK: các kiểm tra loss đều qua")


In [ ]:
# TODO: kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3):
#   1) cố định seed
#   2) loss ban đầu của head mới xấp xỉ -ln(1/9) = 2.197
#   3) overfit được một batch nhỏ tới loss gần 0
#   4) vẽ vài ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn để chắc ảnh và nhãn khớp nhau

## Ngân sách GPU: đo 1 epoch mỗi backbone
Mỗi backbone chạy **1 epoch train + 1 lượt val** với công thức nền (batch 64, AMP, 224). Không ghi gì ra `runs/`.
Từ đó ước lượng tổng thời gian cho kế hoạch ~20-25 lần huấn luyện và quyết định ablation trên 1 hay 2 backbone (GUIDE mục 7).
Ghi bảng này vào báo cáo; mọi cắt giảm (epoch, số backbone ablation...) cũng phải ghi lại.

In [ ]:
import importlib, train as TR
importlib.reload(TR)
from train import Config, measure_epoch_time

BUDGET_BACKBONES = ["resnet50", "resnext50", "convnext_tiny", "deit_small", "swin_tiny", "efficientnet_b0", "mobilenetv3"]
timing = []
for bb in BUDGET_BACKBONES:
    r = measure_epoch_time(Config(backbone=bb))          # đủ 1 epoch; max_batches=40 để đo nhanh (ngoại suy)
    timing.append(r)
    print(f"{bb:16s} train {r['train_epoch_s']:6.0f}s | val {r['val_s']:5.0f}s | mem {r['peak_mem_GB']:.1f} GB")
timing = pd.DataFrame(timing).set_index("backbone")
timing.to_csv("eval_out/epoch_timing.csv")             # lưu ra Drive
display(timing.round(1))


In [ ]:
# Kế hoạch: sửa số lần chạy cho đúng ý bạn, rồi so với hạn mức GPU (Colab free: không cố định, thường vài giờ/phiên).
EPOCHS = 12
ABLATION_BACKBONES = ["resnet50"]          # 1 backbone; thêm cái thứ 2 nếu tổng giờ còn dư
N_ABLATION_RUNS = 12                       # trục A-G, mỗi lần khác T00 một yếu tố
FINAL_SEEDS = 3                            # F01 x 3 seed + mốc T00 thêm 2 seed (seed0 dùng lại từ Bước 2)

ep = timing["epoch_s"] / 3600              # giờ / epoch
hours = {
    "B: backbones x 1 seed": ep[BUDGET_BACKBONES].sum() * EPOCHS,
    "T: ablation": sum(ep[b] for b in ABLATION_BACKBONES) * EPOCHS * N_ABLATION_RUNS,
    "F: chung kết + mốc": ep[ABLATION_BACKBONES[0]] * EPOCHS * (FINAL_SEEDS + FINAL_SEEDS - 1),
}
plan = pd.Series(hours).round(2); plan["TỔNG (giờ GPU)"] = plan.sum(); plan["+20% chạy hỏng"] = plan.iloc[-1] * 1.2
display(plan.to_frame("giờ"))


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# TODO: vòng lặp qua danh sách backbone, mỗi lần gọi train.run(Config(exp_id="B01", backbone=..., seed=0))
#       Gợi ý: from train import Config, run
# TODO: ghi vào bảng: tag trọng số, #params, GMAC, macro-F1 val, top-1 val, thời gian train/epoch.
# TODO: chọn 1-2 backbone đi tiếp, kèm lý do dựa trên số liệu.

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py